# 01 Prevent Analysis

Main PREVENT fairness analysis notebook (Python)

Purpose: run the main model analysis and fairness evaluation, including xCI-related summaries and Kaplan–Meier based checks.

# Set Up Environment

In [ ]:
# import client module with default PySpark output mode
from truveta.study import Client, OutputMode, display_df
client = Client()

# import client module and set PandasOnSpark output mode
from truveta.study import Client, OutputMode, display_df
client = Client(output_mode = OutputMode.PandasOnSpark)

In [ ]:
# get study
study = client.get_study()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, abs as abs_
from pyspark.sql.types import StructType, StructField, FloatType, IntegerType, BooleanType
from sksurv.metrics import (
    concordance_index_censored,
    concordance_index_ipcw,
    cumulative_dynamic_auc,
    integrated_brier_score,
)
from lifelines import KaplanMeierFitter

In [ ]:
saved_df = study.load_artifacts_data("/data/PREVENT_NEW_INDEX_1.parquet")
display_df(saved_df)

In [ ]:
df = saved_df.to_pandas()

# Data Cleaning 

In [ ]:
# Convert index_dtt to year
df['year'] = pd.to_datetime(df['index_dtt']).dt.year

In [ ]:
df['scre'] = df['scre'].astype(float)
df['age'] = df['age'].astype(float)

In [ ]:
scre = df['scre'].to_numpy()
age = df['age'].to_numpy()
female = df['female'].to_numpy()

In [ ]:
egfr_female = 141.57 * np.minimum(scre / 0.7, 1) ** -0.241 * np.maximum(scre / 0.7, 1) ** -1.2 * 0.993839 ** age
egfr_male = 141.57 * np.minimum(scre / 0.9, 1) ** -0.302 * np.maximum(scre / 0.9, 1) ** -1.2 * 0.993839 ** age
df['egfr'] = np.where(female == 0, egfr_male, egfr_female)

In [ ]:
df['fu_ascvd'] = np.minimum(df['fu_mi'], df['fu_str'])
df['fu_cvd'] = np.minimum.reduce([df['fu_mi'], df['fu_str'], df['fu_hf']])

In [ ]:
df['age'] = (df['age'] - 55) / 10
df['nonhdlc'] = (df['chol'] - df['hdlc']) * 0.02586 - 3.5
df['hdlc'] = (df['hdlc'] * 0.02586 - 1.3) / 0.3
df['sbp1'] = (np.minimum(df['sbp'], 110) - 110) / 20
df['sbp2'] = (np.maximum(df['sbp'], 110) - 130) / 20
df['bmi1'] = (np.minimum(df['bmi'], 30) - 25) / 5
df['bmi2'] = (np.maximum(df['bmi'], 30) - 30) / 5
df['egfr1'] = (np.minimum(df['egfr'], 60) - 60) / -15
df['egfr2'] = (np.maximum(df['egfr'], 60) - 90) / -15

In [ ]:
df['sbp2treat'] = df['sbp2'] * df['antihtn']
df['nonhdlctreat'] = df['nonhdlc'] * df['statin']
df['agenonhdlc'] = df['age'] * df['nonhdlc']
df['agehdlc'] = df['age'] * df['hdlc']
df['agesbp2'] = df['age'] * df['sbp2']
df['agediabetes'] = df['age'] * df['diabetes']
df['agecursmk'] = df['age'] * df['cursmk']
df['agebmi2'] = df['age'] * df['bmi2']
df['ageegfr1'] = df['age'] * df['egfr1']
df['cons'] = 1

# Risk from Prevent Coefficient

## ASCVD Risk

### Male

In [ ]:
# Coefficient matrix for males
coefficient_male_ascvd = np.array([
    0.7099847198, 0.1658663005, -0.1144284979, -0.2837212086, 0.3239977062, 0.7189596891,
    0.3956972957, 0.369007498, 0.0203619003, 0.2036522031, -0.0865581036, -0.0322915986,
    0.1145630032, -0.0300005004, 0.0232746992, -0.0927024037, -0.2018525004, -0.0970527008,
    -0.1217081025, -3.500654936
]).reshape(20, 1)

# Subset the data for males (female == 0)
covariate_male_ascvd = df[(df['female'] == 0) ][[
    'age', 'nonhdlc', 'hdlc', 'sbp1', 'sbp2', 'diabetes', 'cursmk', 'egfr1', 'egfr2',
    'antihtn', 'statin', 'sbp2treat', 'nonhdlctreat', 'agenonhdlc', 'agehdlc', 'agesbp2',
    'agediabetes', 'agecursmk', 'ageegfr1', 'cons'
]].values

# Calculate the linear combination
lo_male_ascvd = np.dot(covariate_male_ascvd, coefficient_male_ascvd)

# Calculate the risk
risk_male_ascvd = np.exp(lo_male_ascvd) / (1 + np.exp(lo_male_ascvd))

### Female

In [ ]:
# Coefficient matrix for females
coefficient_female_ascvd = np.array([
    0.7198830247, 0.1176967025, -0.1511850059, -0.0835357979, 0.3592852056, 0.8348584771,
    0.4831078053, 0.4864619076, 0.039777901, 0.2265308946, -0.0592373982, -0.0395761989,
    0.0844423026, -0.0567838997, 0.0325691998, -0.1035984978, -0.241754204, -0.0791141987,
    -0.167149201, -3.819974899
]).reshape(20, 1)

# Subset the data for females (female == 1) 
covariate_female_ascvd = df[(df['female'] == 1)][[
    'age', 'nonhdlc', 'hdlc', 'sbp1', 'sbp2', 'diabetes', 'cursmk', 'egfr1', 'egfr2',
    'antihtn', 'statin', 'sbp2treat', 'nonhdlctreat', 'agenonhdlc', 'agehdlc', 'agesbp2',
    'agediabetes', 'agecursmk', 'ageegfr1', 'cons'
]].values

# Calculate the linear combination
lo_female_ascvd = np.dot(covariate_female_ascvd, coefficient_female_ascvd)

# Calculate the risk (probability)
risk_female_ascvd = np.exp(lo_female_ascvd) / (1 + np.exp(lo_female_ascvd))

## Heart Failure

### Male

In [ ]:
# Coefficient matrix for males (Heart Failure)
coefficient_male_hf = np.array([
    0.8972641826, -0.6811466217, 0.3634460866, 0.9237759709, 0.5023735762, -0.0485840999,
    0.3726929128, 0.6926916838, 0.0251826998, 0.2980921865, -0.0497731008, -0.1289200932,
    -0.3040924072, -0.140168801, 0.0068126, -0.179777801, -3.946391106
]).reshape(17, 1)

# Subset the data for males (female == 0) 
covariate_male_hf = df[(df['female'] == 0)][[
    'age', 'sbp1', 'sbp2', 'diabetes', 'cursmk', 'bmi1', 'bmi2', 'egfr1', 'egfr2',
    'antihtn', 'sbp2treat', 'agesbp2', 'agediabetes', 'agecursmk', 'agebmi2', 'ageegfr1', 'cons'
]].values

# Calculate the linear combination
lo_male_hf = np.dot(covariate_male_hf, coefficient_male_hf)

# Calculate the risk (probability)
risk_male_hf = np.exp(lo_male_hf) / (1 + np.exp(lo_male_hf))

### Female 

In [ ]:
# Coefficient matrix for females (Heart Failure)
coefficient_female_hf = np.array([
    0.8998234868, -0.4559771121, 0.3576504886, 1.038346052, 0.5839160085, -0.0072293999,
    0.2997705936, 0.7451637983, 0.0557086989, 0.3534441888, -0.0981511027, -0.0946663022,
    -0.3581041098, -0.1159453019, -0.003878, -0.1884288937, -4.310409069
]).reshape(17, 1)

# Subset the data for females (female == 1) 
covariate_female_hf = df[(df['female'] == 1)][[
    'age', 'sbp1', 'sbp2', 'diabetes', 'cursmk', 'bmi1', 'bmi2', 'egfr1', 'egfr2',
    'antihtn', 'sbp2treat', 'agesbp2', 'agediabetes', 'agecursmk', 'agebmi2', 'ageegfr1', 'cons'
]].values

# Calculate the linear combination
lo_female_hf = np.dot(covariate_female_hf, coefficient_female_hf)

# Calculate the risk (probability)
risk_female_hf = np.exp(lo_female_hf) / (1 + np.exp(lo_female_hf))

# Evaluate Fairness

## ASCVD

In [ ]:
female_df = df[(df['female'] == 1)]
male_df = df[(df['female'] == 0)]

# Concatenate the DataFrames with females on top and males at the bottom
test_df = pd.concat([female_df, male_df], axis=0).reset_index(drop=True)

# Create female_test and male_test arrays
female_test = (test_df['female'] == 1.0).astype(int).values
male_test = (test_df['female'] == 0.0).astype(int).values

# Create s_ascvd_test array indicating whether an ASCVD event (either MI or stroke) occurred
s_ascvd_test = ((test_df['event_mi'] == 1) | (test_df['event_str'] == 1)).astype(int).values

# Create t_ascvd_test array with follow-up times for ASCVD
t_ascvd_test = test_df['fu_ascvd'].values

# Combine risk_female_ascvd and risk_male_ascvd into one array
combined_risk_ascvd = np.concatenate((risk_female_ascvd, risk_male_ascvd))
combined_risk_ascvd = combined_risk_ascvd.ravel()

### xCI 

In [ ]:
def xCI_spark(s_test, t_test, pred_risk,
              weights=None,
              pos_group=None, neg_group=None,
              return_num_valid=False,
              tied_tol=1e-8):
    
    
    # Initialize SparkSession if not already available
    spark = SparkSession.builder.getOrCreate()
    
    # Convert NumPy arrays to Python lists
    s_test_list = s_test.astype(int).tolist()
    t_test_list = t_test.astype(float).tolist()
    pred_risk_list = pred_risk.astype(float).tolist()
    
    # Handle weights if provided
    if weights is not None:
        weights_list = weights.astype(float).tolist()
    else:
        weights_list = [1.0] * len(s_test_list)
    
    # Convert pos_group and neg_group if provided
    if pos_group is not None:
        pos_group_list = pos_group.astype(bool).tolist()
    else:
        pos_group_list = [True] * len(s_test_list)
    
    if neg_group is not None:
        neg_group_list = neg_group.astype(bool).tolist()
    else:
        neg_group_list = [True] * len(s_test_list)
    
    # Create a list of tuples representing each row
    data = list(zip(s_test_list, t_test_list, pred_risk_list,
                    weights_list, pos_group_list, neg_group_list))
    
    # Define the schema explicitly
    schema = StructType([
        StructField('id', IntegerType(), False),
        StructField('s_test', IntegerType(), True),
        StructField('t_test', FloatType(), True),
        StructField('pred_risk', FloatType(), True),
        StructField('weights', FloatType(), True),
        StructField('pos_group', BooleanType(), True),
        StructField('neg_group', BooleanType(), True)
    ])
    
    # Add unique IDs to each row for joining purposes
    data_with_id = [(i,) + row for i, row in enumerate(data)]
    
    # Create the Spark DataFrame
    df = spark.createDataFrame(data_with_id, schema=schema)
    
    # Prepare the positive group DataFrame
    df_pos = df.filter((col('s_test') == 1) & (col('pos_group') == True))
    df_pos = df_pos.select(col('id').alias('id_pos'),
                           col('t_test').alias('t_test_pos'),
                           col('pred_risk').alias('pred_risk_pos'),
                           col('weights').alias('weights_pos'))
    
    # Prepare the negative group DataFrame
    df_neg = df.filter(col('neg_group') == True)
    df_neg = df_neg.select(col('id').alias('id_neg'),
                           col('t_test').alias('t_test_neg'),
                           col('pred_risk').alias('pred_risk_neg'),
                           col('weights').alias('weights_neg'))
    
    # Perform cross join between positive and negative groups
    df_pairs = df_pos.crossJoin(df_neg)
    
    # Apply the time condition
    df_valid = df_pairs.filter(col('t_test_pos') < col('t_test_neg'))
    
    # Compute weight product
    df_valid = df_valid.withColumn('weight_product', col('weights_pos') * col('weights_neg'))
    
    # Compute risk differences
    df_valid = df_valid.withColumn('risk_diff', col('pred_risk_pos') - col('pred_risk_neg'))
    
    # Determine correctly ranked and tied
    df_valid = df_valid.withColumn(
        'correctly_ranked',
        when(col('risk_diff') > tied_tol, 1.0).otherwise(0.0)
    )
    df_valid = df_valid.withColumn(
        'tied',
        when(abs_(col('risk_diff')) <= tied_tol, 0.5).otherwise(0.0)
    )
    
    # Compute numerator and denominator for concordance index
    df_valid = df_valid.withColumn('contribution',
                                   (col('correctly_ranked') + col('tied')) * col('weight_product'))
    
    numerator = df_valid.agg({'contribution': 'sum'}).collect()[0][0]
    denominator = df_valid.agg({'weight_product': 'sum'}).collect()[0][0]
    
    ci = numerator / denominator if denominator != 0 else None
    
    if return_num_valid:
        num_valid = df_valid.count()
        return ci, num_valid
    else:
        return ci


In [ ]:
xCI_spark(s_ascvd_test, t_ascvd_test, combined_risk_ascvd, return_num_valid=True)

In [ ]:
s_ascvd_test = s_ascvd_test.astype(bool)
ascvd_ci = concordance_index_censored(s_ascvd_test, t_ascvd_test, combined_risk_ascvd)
print(ascvd_ci[0])

In [ ]:
female_ascvd_ci = concordance_index_censored(s_ascvd_test[0:4055], t_ascvd_test[0:4055], combined_risk_ascvd[0:4055])
print(female_ascvd_ci)

In [ ]:
for g1, l1 in zip([female_test, male_test], ['female', 'male']):

    for g2, l2 in zip([female_test, male_test], ['female', 'male']):

        print('xCI %s-%s = %.4f' % (
            l1, l2,
            xCI_spark(
                s_ascvd_test, t_ascvd_test, combined_risk_ascvd,
                pos_group=g1, neg_group=g2
            )
        ))

### KM

In [ ]:
ascvd = pd.DataFrame({
    't_ascvd': t_ascvd_test,
    's_ascvd': s_ascvd_test.astype(int),
    'pred_prob' : combined_risk_ascvd,
    'female' : female_test
})
print(ascvd)
# Convert Pandas DataFrame to PySpark DataFrame
spark_df = spark.createDataFrame(ascvd)

In [ ]:
# Function to calculate and plot Kaplan-Meier curve
def plot_km_curve_pandas(df, time_col, event_col):
    """
    Calculate and plot Kaplan-Meier survival curve using Pandas DataFrame and lifelines.

    :param df: Pandas DataFrame containing the data
    :param time_col: The name of the column containing time to event or censoring
    :param event_col: The name of the column indicating whether the event occurred (1 = event, 0 = censored)
    """
    # Step 1: Initialize KaplanMeierFitter from lifelines
    kmf = KaplanMeierFitter()

    # Step 2: Fit the data (time and event)
    kmf.fit(df[time_col], event_observed=df[event_col])

    # Step 3: Plot the Kaplan-Meier survival curve
    plt.figure(figsize=(8, 6))
    kmf.plot_survival_function()
    plt.title('Kaplan-Meier Survival Curve')
    plt.xlabel('Time')
    plt.ylabel('Survival Probability')
    plt.grid(True)
    plt.show()

# Example Usage:
# Assuming you have a Pandas DataFrame `df` with columns "time_to_event" and "event_occurred"
plot_km_curve_pandas(ascvd, "t_ascvd", "s_ascvd")

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Function to calculate and plot the Kaplan-Meier curve
def plot_km_curve(spark, data, time_col, event_col):
    """
    Calculate and plot Kaplan-Meier survival curve using PySpark.
    
    :param spark: SparkSession object
    :param data: PySpark DataFrame containing the data
    :param time_col: The name of the column containing time to event or censoring
    :param event_col: The name of the column indicating whether the event occurred (1 = event, 0 = censored)
    """
    # Step 1: Sort data by time
    data_sorted = data.orderBy(time_col)

    # Step 2: Add a column for the number at risk (i.e., the number of patients who have not had the event)
    window_spec = Window.orderBy(time_col)
    
    data_sorted = data_sorted.withColumn('n_at_risk', F.count(time_col).over(window_spec))
    
    # Step 3: Calculate the probability of survival at each time step
    data_sorted = data_sorted.withColumn('event_probability', 
                                         F.when(F.col(event_col) == 1, F.lit(1.0)).otherwise(F.lit(0.0)))

    data_sorted = data_sorted.withColumn('survival_probability', 
                                         (F.col('n_at_risk') - F.sum(F.col('event_probability')).over(window_spec)) / F.col('n_at_risk'))
    
    # Step 4: Compute the cumulative survival probability over time
    data_sorted = data_sorted.withColumn('km_curve', 
                                         F.exp(F.sum(F.log(F.col('survival_probability'))).over(window_spec)))
    
    # Step 5: Convert to Pandas DataFrame for plotting
    km_data = data_sorted.select(time_col, 'km_curve').toPandas()
    
    # Step 6: Plot the Kaplan-Meier survival curve
    plt.figure(figsize=(8, 6))
    plt.step(km_data[time_col], km_data['km_curve'], where="post")
    plt.title('Kaplan-Meier Survival Curve')
    plt.xlabel('Time')
    plt.ylabel('Survival Probability')
    plt.grid(True)
    plt.show()


plot_km_curve(spark, spark_df, "t_ascvd", "s_ascvd")

In [ ]:
def xAUCt_pyspark(spark_df, s_test_col, t_test_col, pred_risk_col, times, pos_group_col=None, neg_group_col=None):
    from pyspark.sql import functions as F
    from pyspark.sql.window import Window

    # Create time window in DataFrame
    time_col = F.array([F.lit(t) for t in times])
    df = spark_df.withColumn("times", time_col)

    # Ensure pred_risk is 2d (static or time-varying)
    df = df.withColumn("pred_risk", 
                       F.when(F.col(pred_risk_col).isNull(), F.array(F.col(pred_risk_col)))
                       .otherwise(F.col(pred_risk_col)))

    # Define positive and negative group conditions
    pos_condition = (F.col(t_test_col) <= F.explode(F.col("times"))) & (F.col(s_test_col) == 1)
    if pos_group_col is not None:
        pos_condition = pos_condition & (F.col(pos_group_col) == 1)

    neg_condition = F.col(t_test_col) > F.explode(F.col("times"))
    if neg_group_col is not None:
        neg_condition = neg_condition & (F.col(neg_group_col) == 1)

    # Filter for valid comparisons
    valid = df.filter(pos_condition & neg_condition)

    # Compare correctly ranked entries
    windowSpec = Window.orderBy(F.col(pred_risk_col).desc())
    valid = valid.withColumn("rank", F.row_number().over(windowSpec))

    # Aggregate correctly ranked comparisons
    valid_ranked = valid.withColumn("correctly_ranked", F.when(F.col("rank") == 1, 1).otherwise(0))

    # Calculate xAUCt
    total_correct = valid_ranked.agg(F.sum("correctly_ranked")).collect()[0][0]
    total_valid = valid.agg(F.count("*")).collect()[0][0]

    return total_correct / total_valid if total_valid > 0 else None


In [ ]:
pyspark_df = spark.createDataFrame(test_df)
xAUCt_pyspark(pyspark_df, s_ascvd_test, t_ascvd_test, combined_risk_ascvd, [1,2,3], pos_group_col=None, neg_group_col=None)

## Heart Failure

In [ ]:
# Create s_ascvd_test array indicating whether an Heart Faliure(HF) occurred
s_hf_test = ((test_df['event_hf'] == 1)).astype(int).values

# Create t_ascvd_test array with follow-up times for HF
t_hf_test = test_df['fu_hf'].values

# Combine risk_female_ascvd and risk_male_ascvd into one array
combined_risk_hf = np.concatenate((risk_female_hf, risk_male_hf))
combined_risk_hf = combined_risk_hf.ravel()

In [ ]:
xCI_spark(s_hf_test, t_hf_test, combined_risk_hf, return_num_valid=True)

In [ ]:
s_hf_test = s_hf_test.astype(bool)
hf_ci = concordance_index_censored(s_hf_test, t_hf_test, combined_risk_hf)
print(hf_ci)

In [ ]:
for g1, l1 in zip([female_test, male_test], ['female', 'male']):

    for g2, l2 in zip([female_test, male_test], ['female', 'male']):

        print('xCI %s-%s = %.4f' % (
            l1, l2,
            xCI_spark(
                s_hf_test, t_hf_test, combined_risk_hf,
                pos_group=g1, neg_group=g2
            )
        ))